# yt-clips Kaggle Worker
## Settings → Accelerator → GPU × 2 (T4)
Run all cells. Worker listens for jobs from your Mac.
Add host reference photos to the `photos/` folder for automatic face detection & tracking.

In [ ]:
# CELL 1: Check GPU + Mount Kaggle Dataset
import subprocess, os, sys
from pathlib import Path

# Check GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu or 'NONE! Enable GPU × 2 in Settings → Accelerator'}")

# Kaggle datasets are mounted at /kaggle/input/
# You can add your yt-clips repo as a dataset:
# 1. Zip your repo: zip -r yt-clips.zip yt-clips/
# 2. Upload to Kaggle Datasets
# 3. Add dataset in notebook settings

# Try common mount points
for p in ["/kaggle/working", "/kaggle/input/yt-clips"]:
    if Path(p).exists():
        os.chdir(p if p != "/kaggle/input/yt-clips" else p)
        print(f"Working dir: {p}")
        break
else:
    os.chdir("/kaggle/working")
    print("Working dir: /kaggle/working")

# Clone repo if not present
if not Path("pipeline.py").exists():
    print("Cloning yt-clips repo...")
    subprocess.run("git clone --depth 1 https://github.com/fearless16/yt-clips.git _repo", shell=True)
    # Move contents from _repo to current dir
    import shutil
    for item in Path("_repo").iterdir():
        dest = Path(item.name)
        if dest.exists():
            if dest.is_dir():
                shutil.rmtree(dest)
            else:
                dest.unlink()
        shutil.move(str(item), ".")
    shutil.rmtree("_repo")

print(f"CWD: {os.getcwd()}")

In [ ]:
# CELL 2: Install System Dependencies
def run(cmd, desc=""):
    print(f"  -> {desc}...") if desc else None
    subprocess.run(cmd, shell=True, capture_output=True)

print("Installing system deps...")
run("apt-get update -qq > /dev/null 2>&1", "apt update")
run("apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1", "aria2 + ffmpeg")
run("npm install -g localtunnel > /dev/null 2>&1", "localtunnel")
print("  ✅ System deps installed")

In [ ]:
# CELL 3: Install Python Dependencies
print("Installing Python deps...")
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
    "filterpy scipy google-genai google-generativeai openai python-dotenv "
    "ultralytics torch face_recognition "
    "'realesrgan>=0.3.0' 'basicsr>=1.4.2' 'gfpgan>=1.3.5' 'facexlib>=0.3.0' "
    "--extra-index-url https://download.pytorch.org/whl/cu121",
    "Python + PyTorch + YOLO + FaceRec + RealESRGAN + GFPGAN")

# Install CodeFormer
print("  -> CodeFormer...")
if not Path("CodeFormer").exists():
    run("git clone https://github.com/sczhou/CodeFormer.git", "Cloning CodeFormer")
    run("cd CodeFormer && pip install -r requirements.txt", "CodeFormer deps")
    run("cd CodeFormer && python basicsr/setup.py develop", "CodeFormer setup")
    run("cd CodeFormer && python scripts/download_pretrained_models.py facelib", "Face models")
    run("cd CodeFormer && python scripts/download_pretrained_models.py CodeFormer", "CodeFormer model")
print("  ✅ CodeFormer installed")

# Install RIFE
print("  -> RIFE...")
if not Path("ECCV2022-RIFE").exists():
    run("git clone https://github.com/hzwer/ECCV2022-RIFE.git", "Cloning RIFE")
    run("cd ECCV2022-RIFE && pip install -r requirements.txt", "RIFE deps")
print("  ✅ RIFE installed")

# Install Video2X (all-in-one pipeline)
print("  -> Video2X...")
run("pip install -q video2x", "Video2X")

# Install DeepFace (best face detection)
print("  -> DeepFace...")
run("pip install -q deepface tf-keras", "DeepFace")

print("  ✅ All Python deps installed")

In [ ]:
# CELL 4: Verify GPU + Models
import torch

print("=" * 55)
print("  GPU STATUS")
print("=" * 55)
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

# Verify models
print("\n" + "=" * 55)
print("  MODEL STATUS")
print("=" * 55)

# Real-ESRGAN
try:
    from realesrgan import RealESRGANer
    print("  ✅ Real-ESRGAN: available")
except ImportError:
    print("  ❌ Real-ESRGAN: not installed")

# GFPGAN
try:
    from gfpgan import GFPGANer
    print("  ✅ GFPGAN: available")
except ImportError:
    print("  ❌ GFPGAN: not installed")

# CodeFormer
try:
    sys.path.insert(0, "CodeFormer")
    print("  ✅ CodeFormer: available")
except:
    print("  ❌ CodeFormer: not found")

# RIFE
try:
    sys.path.insert(0, "ECCV2022-RIFE")
    print("  ✅ RIFE: available")
except:
    print("  ❌ RIFE: not found")

# DeepFace
try:
    from deepface import DeepFace
    print("  ✅ DeepFace: available")
except ImportError:
    print("  ❌ DeepFace: not installed")

# Video2X
try:
    import video2x
    print("  ✅ Video2X: available")
except ImportError:
    print("  ❌ Video2X: not installed")

print("\n  Ready for AI pipeline!")

In [ ]:
# CELL 5: Apply Torchvision Compat Shim
import utils.torchvision_compat
print("  ✅ torchvision compat shim applied")

# Create folders
for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs", "photos"]:
    Path(folder).mkdir(exist_ok=True)

In [ ]:
# CELL 6: Start Worker + Tunnel
import time

subprocess.run("pkill -f 'python watcher.py' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'lt --port' 2>/dev/null || true", shell=True)
time.sleep(1)

subprocess.Popen([sys.executable, "watcher.py"], stdout=open("watcher.log","w"), stderr=subprocess.STDOUT)
time.sleep(2)
subprocess.Popen(["lt", "--port", "5000"], stdout=open("tunnel.log","w"), stderr=subprocess.STDOUT)
time.sleep(5)

with open("tunnel.log") as f:
    for line in f:
        if "://" in line.strip():
            with open("kaggle_url.txt", "w") as out:
                out.write(line.strip())
            print(f"Tunnel URL: {line.strip()}")
            break

In [ ]:
# CELL 7: Monitor (keep this cell running)
print("=" * 55)
print("  WORKER IS ONLINE!")
print("=" * 55)
print("On Mac, run:")
print('  ./automate.sh "URL"')
print("  -> Option 2 (Remote Run)")

try:
    last_pos = 0
    last_inode = None
    while True:
        time.sleep(10)
        try:
            st = Path("watcher.log").stat()
        except FileNotFoundError:
            continue
        if last_inode is not None and st.st_ino != last_inode:
            last_pos = 0
        last_inode = st.st_ino
        with open("watcher.log", "r") as f:
            f.seek(last_pos)
            for l in f.readlines():
                if l.strip():
                    print(l.strip())
            last_pos = f.tell()
except KeyboardInterrupt:
    print("Stopped.")